In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

project_path = '/content/drive/MyDrive/Spacecraft-Anomaly-Detection'

data_path = project_path + '/data/raw/archive/data/data'
train_path = data_path + '/train'
test_path = data_path + '/test'
labels_path = project_path + '/data/raw/archive/labeled_anomalies.csv'

results_path = project_path + '/results'

print("SERIAL 17 loaded successfully.")
print("Project folder exists:", os.path.exists(project_path))
print("Train folder exists:", os.path.exists(train_path))
print("Test folder exists:", os.path.exists(test_path))
print("Labels file exists:", os.path.exists(labels_path))
print("Results folder exists:", os.path.exists(results_path))

Mounted at /content/drive
SERIAL 17 loaded successfully.
Project folder exists: True
Train folder exists: True
Test folder exists: True
Labels file exists: True
Results folder exists: True


#####Load A-8 Data and Ground Truth

Now we'll load the same A-8 training/test data and the actual anomaly labels.

In [2]:
channel = "A-8"

# Load training and test data
train_A8 = np.load(train_path + "/" + channel + ".npy")
test_A8 = np.load(test_path + "/" + channel + ".npy")

# Use the first column as telemetry
train_telemetry_A8 = train_A8[:, 0]
test_telemetry_A8 = test_A8[:, 0]

# Load anomaly labels
labels_df = pd.read_csv(labels_path)
A8_info = labels_df[labels_df["chan_id"] == channel].iloc[0]

print("Channel:", channel)
print("Training data shape:", train_A8.shape)
print("Test data shape:", test_A8.shape)
print("Ground-truth anomaly:", A8_info["anomaly_sequences"])
print("Anomaly class:", A8_info["class"])

Channel: A-8
Training data shape: (762, 25)
Test data shape: (8375, 25)
Ground-truth anomaly: [[4569, 8374]]
Anomaly class: [contextual]


Create Ground-Truth Labels

We'll now convert that anomaly sequence into a binary label array:

0 = normal
1 = anomaly

In [3]:
# Create ground-truth labels
y_true_A8 = np.zeros(len(test_telemetry_A8), dtype=int)

anomaly_start = 4569
anomaly_end = 8374

y_true_A8[anomaly_start:anomaly_end + 1] = 1

print("Ground-truth labels created.")
print("Ground-truth normal:", (y_true_A8 == 0).sum())
print("Ground-truth anomalies:", (y_true_A8 == 1).sum())
print("Total test points:", len(y_true_A8))

Ground-truth labels created.
Ground-truth normal: 4569
Ground-truth anomalies: 3806
Total test points: 8375


Recreate Isolation Forest Predictions

Now we'll recreate the exact Isolation Forest setup from SERIAL 16 so our evaluation is reproducible.

In [4]:
# Prepare training and test data
X_train_A8 = train_telemetry_A8.reshape(-1, 1)
X_test_A8 = test_telemetry_A8.reshape(-1, 1)

# Create the same Isolation Forest model used in SERIAL 16
isolation_forest = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42
)

# Train using training data only
isolation_forest.fit(X_train_A8)

# Predict on test data
isolation_predictions = isolation_forest.predict(X_test_A8)

# Convert:
# 1 = normal → 0
# -1 = anomaly → 1
y_pred_isolation_A8 = (isolation_predictions == -1).astype(int)

print("Isolation Forest predictions recreated.")
print("Predicted normal:", (y_pred_isolation_A8 == 0).sum())
print("Predicted anomalies:", (y_pred_isolation_A8 == 1).sum())

Isolation Forest predictions recreated.
Predicted normal: 0
Predicted anomalies: 8375


Calculate Precision, Recall & F1

Now we'll compare:

y_true_A8 → actual NASA labels

y_pred_isolation_A8 → Isolation Forest predictions

In [5]:
# Calculate evaluation metrics
precision = precision_score(
    y_true_A8,
    y_pred_isolation_A8,
    zero_division=0
)

recall = recall_score(
    y_true_A8,
    y_pred_isolation_A8,
    zero_division=0
)

f1 = f1_score(
    y_true_A8,
    y_pred_isolation_A8,
    zero_division=0
)

print("Isolation Forest Evaluation")
print("---------------------------")
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

Isolation Forest Evaluation
---------------------------
Precision: 0.45444776119402985
Recall: 1.0
F1-score: 0.6249076430506526


| Metric    | Isolation Forest |
| --------- | ---------------: |
| Precision |       **0.4544** |
| Recall    |       **1.0000** |
| F1-score  |       **0.6249** |


#Confusion Matrix

In [6]:
# Calculate confusion matrix
tn, fp, fn, tp = confusion_matrix(
    y_true_A8,
    y_pred_isolation_A8
).ravel()

print("Confusion Matrix")
print("-----------------")
print("True Negatives (TN):", tn)
print("False Positives (FP):", fp)
print("False Negatives (FN):", fn)
print("True Positives (TP):", tp)

Confusion Matrix
-----------------
True Negatives (TN): 0
False Positives (FP): 4569
False Negatives (FN): 0
True Positives (TP): 3806


| Metric    |     Result |
| --------- | ---------: |
| TN        |          0 |
| FP        |  **4,569** |
| FN        |          0 |
| TP        |  **3,806** |
| Precision | **45.44%** |
| Recall    |   **100%** |
| F1        | **62.49%** |


###Calculate FPR and FNR

In [7]:
# Calculate false positive and false negative rates
false_positive_rate = fp / (fp + tn)
false_negative_rate = fn / (fn + tp)

print("False Positive Rate (FPR):", false_positive_rate)
print("False Negative Rate (FNR):", false_negative_rate)

False Positive Rate (FPR): 1.0
False Negative Rate (FNR): 0.0


##Research interpretation

The baseline Isolation Forest successfully detected all ground-truth anomalies, but it also classified all normal observations as anomalies.

So the main problem is:

Extremely high false-alarm rate.

This means it is not practical as-is for spacecraft anomaly detection, where thousands of false alarms could overwhelm operators.

But this result gives us a useful research direction: we need better features and/or a better-calibrated anomaly threshold.

##Save the Evaluation Result

In [8]:
# Save Isolation Forest evaluation results

isolation_result = pd.DataFrame([{
    "channel": channel,
    "method": "Isolation Forest",
    "contamination": "auto",
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "false_positive_rate": false_positive_rate,
    "false_negative_rate": false_negative_rate,
    "true_negatives": tn,
    "false_positives": fp,
    "false_negatives": fn,
    "true_positives": tp
}])

output_file = results_path + "/isolation_forest_evaluation_A8.csv"

isolation_result.to_csv(output_file, index=False)

print("Evaluation results saved.")
print("File:", output_file)

display(isolation_result)

Evaluation results saved.
File: /content/drive/MyDrive/Spacecraft-Anomaly-Detection/results/isolation_forest_evaluation_A8.csv


,channel,method,contamination,precision,recall,f1_score,false_positive_rate,false_negative_rate,true_negatives,false_positives,false_negatives,true_positives
0,A-8,Isolation Forest,auto,0.454448,1.0,0.624908,1.0,0.0,0,4569,0,3806


### Research conclusion

The baseline Isolation Forest detected all ground-truth anomalies on A-8, achieving 100% recall, but classified every normal observation as anomalous, resulting in a 100% false-positive rate. This indicates that the raw one-dimensional Isolation Forest with its default threshold is overly sensitive and unsuitable for practical anomaly detection on this channel.

#github commit

In [9]:
!git add notebooks/17_evaluate_isolation_forest.ipynb results/isolation_forest_evaluation_A8.csv

fatal: not a git repository (or any of the parent directories): .git


In [10]:
!git commit -m "Complete SERIAL 17 Isolation Forest evaluation"

fatal: not a git repository (or any of the parent directories): .git


In [11]:
!git push origin main

fatal: not a git repository (or any of the parent directories): .git


In [12]:
%cd /content/drive/MyDrive/Spacecraft-Anomaly-Detection

!git status

/content/drive/MyDrive/Spacecraft-Anomaly-Detection
Refresh index: 100% (23/23), done.
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/13_evaluate_global_zscore.ipynb
	modified:   notebooks/14_rolling_zscore.ipynb
	modified:   notebooks/15_compare_statistical_methods.ipynb
	modified:   notebooks/16_isolation_forest.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/17_evaluate_isolation_forest.ipynb
	results/isolation_forest_evaluation_A8.csv

no changes added to commit (use "git add" and/or "git commit -a")


In [13]:
!git add notebooks/17_evaluate_isolation_forest.ipynb results/isolation_forest_evaluation_A8.csv

In [14]:
!git commit -m "Complete SERIAL 17 Isolation Forest evaluation"

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@1f666b13bdf2.(none)')


In [15]:
!git config --global user.name "Amit Chandra Das"
!git config --global user.email "arickroy0@gmail.com"

In [16]:
!git push origin main

Everything up-to-date


In [17]:
!git commit -m "Complete SERIAL 17 Isolation Forest evaluation"

[main 1198d43] Complete SERIAL 17 Isolation Forest evaluation
 2 files changed, 3 insertions(+)
 create mode 100644 notebooks/17_evaluate_isolation_forest.ipynb
 create mode 100644 results/isolation_forest_evaluation_A8.csv


In [18]:
!git push origin main

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 6.67 KiB | 1.11 MiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Amit-Chandra-Das/spacecraft-anomaly-detection.git
   ef43bb5..1198d43  main -> main
